In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import scipy.stats as stats

PROJECT_ROOT = Path.cwd()   # open this notebook from D:\upi-failure-economics
DATA_FILE = PROJECT_ROOT / "data" / "processed" / "npci_upi_clean_data.csv"

df = pd.read_csv(DATA_FILE)
print("Loaded:", DATA_FILE)
print(df.shape)
df.head()

Loaded: d:\upi-failure-economics\data\processed\npci_upi_clean_data.csv
(3000, 9)


,month_year,bank_name,role_type,total_volume_mn,approved_volume_mn,approved_pct,technical_decline_pct,business_decline_pct,total_decline_pct
0,2024-04,State Bank Of India,Remitter,3581.27,3369.98,94.10,0.89,5.01,5.90
1,2024-04,HDFC BANK LTD,Remitter,1157.04,1108.68,95.82,0.09,4.08,4.17
2,2024-04,Bank of Baroda,Remitter,898.17,811.86,90.39,1.59,8.02,9.61
3,2024-04,Union Bank of India,Remitter,837.24,779.72,93.13,1.59,5.28,6.87
4,2024-04,Punjab National Bank,Remitter,741.90,689.67,92.96,1.11,5.93,7.04


In [2]:
print("=" * 50)
print("=== 1. VOLUME STRESS CORRELATION (Top 10 Banks) ===")
print("=" * 50)

remitters = df[df['role_type'] == 'Remitter']

# Top 10 busiest remitter banks by total volume
top_banks = remitters.groupby('bank_name')['total_volume_mn'].sum().nlargest(10).index

corr_results = []
for bank in top_banks:
    bank_data = remitters[remitters['bank_name'] == bank]
    # Need enough points and non-constant values for Pearson to work
    if len(bank_data) > 5 and bank_data['technical_decline_pct'].nunique() > 1:
        r, p = stats.pearsonr(bank_data['total_volume_mn'], bank_data['technical_decline_pct'])
        corr_results.append({'bank_name': bank, 'months': len(bank_data),
                             'pearson_r': round(r, 2), 'p_value': round(p, 4)})

corr_df = pd.DataFrame(corr_results)
corr_df['significant (p<0.05)'] = corr_df['p_value'] < 0.05
corr_df

=== 1. VOLUME STRESS CORRELATION (Top 10 Banks) ===


,bank_name,months,pearson_r,p_value,significant (p<0.05)
0,State Bank of India,18,-0.18,0.4791,False
1,State Bank Of India,12,-0.28,0.3762,False
2,Bank of Baroda,27,-0.46,0.0154,True
3,HDFC Bank Ltd.,19,-0.24,0.3176,False
4,Punjab National Bank,27,0.16,0.4242,False
5,Union Bank of India,24,-0.65,0.0005,True
6,Kotak Mahindra Bank,29,-0.65,0.0001,True
7,Canara Bank,30,-0.45,0.0127,True
8,Axis Bank Ltd.,28,-0.33,0.0889,False
9,Indian Bank,30,0.07,0.7249,False


In [3]:
print("=" * 50)
print("=== 2. Z-SCORE ANOMALY DETECTION (Spikes z > 2.5) ===")
print("=" * 50)

def zscore(x):
    std = x.std()
    return (x - x.mean()) / (std if std > 0 else 1)

# Z-score within each bank AND role
df['td_zscore'] = df.groupby(['bank_name', 'role_type'])['technical_decline_pct'].transform(zscore)

anomalies = (df[df['td_zscore'] > 2.5]
             [['month_year', 'bank_name', 'role_type', 'technical_decline_pct', 'td_zscore']]
             .sort_values('td_zscore', ascending=False))

print(f"Total severe anomaly months flagged (z > 2.5): {len(anomalies)}")
anomalies.head(10)

=== 2. Z-SCORE ANOMALY DETECTION (Spikes z > 2.5) ===
Total severe anomaly months flagged (z > 2.5): 79


,month_year,bank_name,role_type,technical_decline_pct,td_zscore
517,2024-01,UCO Bank,Remitter,4.16,4.772319
690,2024-07,Maharashtra Gramin Bank,Remitter,11.07,4.717503
245,2025-08,DBS Bank India Limited,Remitter,1.48,4.682965
2962,2025-09,Indian Bank,Beneficiary,1.12,4.540380
2019,2024-01,UCO Bank,Beneficiary,3.24,4.333730
2122,2026-01,Fino Payments Bank Ltd.,Beneficiary,6.95,4.251079
925,2024-03,Karur Vysya Bank,Remitter,2.51,4.245318
411,2025-02,Airtel Payments Bank Limited,Remitter,2.41,4.136889
1714,2025-08,Bank Of India,Beneficiary,4.07,4.119305
1711,2025-08,Federal Bank Ltd.,Beneficiary,0.44,4.103825


In [4]:
print("Anomalies by role:")
print(anomalies['role_type'].value_counts(), "\n")

print("Banks with the most anomaly months:")
print(anomalies.groupby(['bank_name', 'role_type']).size().sort_values(ascending=False).head(10), "\n")

print("Anomalies by month:")
print(anomalies['month_year'].value_counts().sort_index())

Anomalies by role:
role_type
Beneficiary    43
Remitter       36
Name: count, dtype: int64 

Banks with the most anomaly months:
bank_name                         role_type  
Axis Bank Ltd.                    Beneficiary    2
                                  Remitter       2
Rajasthan Marudhara Gramin Bank   Beneficiary    2
The Jammu & Kashmir Bank Limited  Beneficiary    2
Kotak Mahindra Bank               Beneficiary    2
Maharashtra Gramin Bank           Beneficiary    2
Bank of Maharashtra               Beneficiary    2
Kotak Mahindra Bank               Remitter       2
Bank Of India                     Beneficiary    1
Airtel Payments Bank Limited      Remitter       1
dtype: int64 

Anomalies by month:
month_year
2024-01     9
2024-02     3
2024-03     4
2024-04     8
2024-06     1
2024-07     9
2024-08     1
2024-09     1
2024-10     2
2024-11     3
2024-12     2
2025-02     3
2025-03     2
2025-04     1
2025-06     1
2025-08    14
2025-09     4
2025-10     1
2025-11     2
202

In [6]:
out_dir = PROJECT_ROOT / "sql_query_result_npci"
out_dir.mkdir(exist_ok=True)

corr_df.to_csv(out_dir / "volume_stress_correlation.csv", index=False)
anomalies.to_csv(out_dir / "td_zscore_anomalies.csv", index=False)
print("Saved to:", out_dir)

Saved to: d:\upi-failure-economics\sql_query_result_npci
